# Global Classifier Comparison — 154-Class ASL Recognition
**Author:** Owasu Brown | National University | 2025–2026

Compares four classifiers on the 154-class ASL experiment.
All tables follow APA 7 formatting conventions.

## Classifiers
- Random Forest (RF)
- Logistic Regression (LR)
- Canonical Interval Forest (CIF)
- InceptionTime (IT)

## Run order
Cell 0 (mount) → 1 (install) → 2 (imports & paths) → 3 (registry) → 4 (load artifacts) → 5 (APA table) → 6–8 (figures) → 9 (export)


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install',
    'pandas', 'numpy', 'scikit-learn', 'matplotlib', 'joblib', '-q'
])
print('Packages ready')


Packages ready


In [3]:
import os, sys, json, warnings
import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.metrics import (
    accuracy_score, f1_score, top_k_accuracy_score)
warnings.filterwarnings('ignore')

PROJECT_DIR    = '/content/drive/MyDrive/Consultant/Colab_Notebooks/Obrown_Dissertation_NU_25/OBrown_DIS9300_v2'
RESULTS_DIR    = os.path.join(PROJECT_DIR, 'results')
COMPARISON_DIR = '/content/drive/MyDrive/Consultant/Colab_Notebooks/Obrown_Dissertation_NU_25/OBrown_DIS9300_v2/results/end_point/global_comparison_v2'
os.makedirs(COMPARISON_DIR, exist_ok=True)

# ── APA 7 style ──────────────────────────────────────────────────────────
MODEL_COLORS = {
    'Random Forest'      : '#2E75B6',
    'Logistic Regression': '#ED7D31',
    'CIF'                : '#70AD47',
    'InceptionTime'      : '#7030A0',
}
MODEL_MARKERS = {
    'Random Forest'      : 'o',
    'Logistic Regression': 's',
    'CIF'                : '^',
    'InceptionTime'      : 'D',
}
MODEL_ORDER = ['InceptionTime', 'Random Forest', 'Logistic Regression', 'CIF']

plt.rcParams.update({
    'font.family'      : 'serif',
    'font.size'        : 10,
    'axes.titlesize'   : 11,
    'axes.labelsize'   : 10,
    'legend.fontsize'  : 9,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.grid'        : False,
    'figure.dpi'       : 150,
})
print(f'Output dir : {COMPARISON_DIR}')


Output dir : /content/drive/MyDrive/Consultant/Colab_Notebooks/Obrown_Dissertation_NU_25/OBrown_DIS9300_v2/results/end_point/global_comparison_v2


In [4]:
# ── 154-class model registry ─────────────────────────────────────────────
#
#  Model               | Metrics source                   | Proba source
#  ────────────────────┼──────────────────────────────────┼──────────────────────────
#  InceptionTime       | inc_154_metrics.json             | inc_154_proba_test.npy
#  Random Forest       | rf_154_metrics.json              | rf_154_proba_test.npy
#  Logistic Regression | lr_154_metrics.json              | lr_154_proba_test.npy
#  CIF                 | cif_154_metrics.json (or similar)| cif_154_proba_test.npy

P = PROJECT_DIR
R = RESULTS_DIR

REGISTRY = [
    dict(model='InceptionTime',
         result_dir='inc_154',
         metrics_tag='inc_154',
         proba_tag='inc_154',
         pkl_name=None,          # saved as .keras, size reported separately
         keras_path=f'{R}/inc_154/best_model_seed42.keras'),

    dict(model='Random Forest',
         result_dir='rf_154',
         metrics_tag='rf_154',
         proba_tag='rf_154',
         pkl_name='rf_154_model.pkl',
         keras_path=None),

    dict(model='Logistic Regression',
         result_dir='lr_154',
         metrics_tag='lr_154',
         proba_tag='lr_154',
         pkl_name='lr_154_model.pkl',
         keras_path=None),

    dict(model='CIF',
         result_dir='cif_154',
         metrics_tag='cif_154',
         proba_tag='cif_154',
         pkl_name='cif_model.pkl',
         keras_path=None),
]

print('Verifying files...')
for slot in REGISTRY:
    base = os.path.join(R, slot['result_dir'])
    mf   = os.path.join(base, f"{slot['metrics_tag']}_metrics.json")
    pf   = os.path.join(base, f"{slot['proba_tag']}_proba_test.npy")
    mok  = '✅' if os.path.exists(mf) else '❌ MISSING metrics'
    pok  = '✅' if os.path.exists(pf) else '⚠  no proba'
    print(f'  {slot["model"]:25s}  metrics:{mok}  proba:{pok}')


Verifying files...
  InceptionTime              metrics:✅  proba:✅
  Random Forest              metrics:✅  proba:✅
  Logistic Regression        metrics:✅  proba:✅
  CIF                        metrics:✅  proba:✅


In [5]:
records = []

def find_file(base, suffix):
    for root, _, files in os.walk(base):
        for f in files:
            if f.endswith(suffix):
                return os.path.join(root, f)
    return None

for slot in REGISTRY:
    base = os.path.join(R, slot['result_dir'])

    # ── Load metrics JSON ─────────────────────────────────────────────────
    mpath = os.path.join(base, f"{slot['metrics_tag']}_metrics.json")
    if not os.path.exists(mpath):
        print(f'  SKIP {slot["model"]} — metrics JSON not found')
        continue
    with open(mpath) as f:
        gm = json.load(f)

    # ── Load proba for Top-5 ──────────────────────────────────────────────
    ppath = os.path.join(base, f"{slot['proba_tag']}_proba_test.npy")
    y_proba = np.load(ppath, allow_pickle=True) if os.path.exists(ppath) else None

    # ── Load predictions CSV for per-class detail ─────────────────────────
    pred_path = find_file(base, '_predictions.csv')
    if pred_path:
        df_pred = pd.read_csv(pred_path)
        y_true  = df_pred['y_true'].values
        y_pred  = df_pred['y_pred'].values
    else:
        y_true = y_pred = None

    # ── Model file size ───────────────────────────────────────────────────
    if slot['keras_path'] and os.path.exists(slot['keras_path']):
        size_mb = os.path.getsize(slot['keras_path']) / (1024**2)
    elif slot['pkl_name']:
        pkl = os.path.join(base, slot['pkl_name'])
        size_mb = os.path.getsize(pkl) / (1024**2) if os.path.exists(pkl) else None
    else:
        size_mb = None

    # ── Top-5 from proba ──────────────────────────────────────────────────
    top5 = gm.get('top5_acc', None)
    if top5 is None and y_proba is not None and y_true is not None:
        try:
            n_cls = y_proba.shape[1]
            top5 = top_k_accuracy_score(
                y_true, y_proba, k=5, labels=list(range(n_cls)))
        except Exception:
            top5 = None

    records.append({
        'Model'       : slot['model'],
        'Accuracy'    : gm.get('top1_acc', gm.get('accuracy', None)),
        'Top-5 Acc'   : top5,
        'F1 Macro'    : gm.get('f1_macro', None),
        'F1 Weighted' : gm.get('f1_weighted', None),
        'Precision'   : gm.get('precision_macro', None),
        'Recall'      : gm.get('recall_macro', None),
        'AUC'         : gm.get('macro_auc', None),
        'N Classes'   : gm.get('n_classes', 154),
        'N Test'      : gm.get('n_test', None),
        'Model MB'    : round(size_mb, 2) if size_mb else None,
        '_y_true'     : y_true,
        '_y_pred'     : y_pred,
        '_y_proba'    : y_proba,
    })
    acc = records[-1]['Accuracy']
    print(f'  ✅  {slot["model"]:25s}  '
          f'Top-1={acc*100:.2f}%  '
          f'Top-5={top5*100:.2f}%' if top5 else f'  ✅  {slot["model"]:25s}  Top-1={acc*100:.2f}%')

df_all = pd.DataFrame(records)
df_all_ordered = df_all.set_index('Model').reindex(
    [m for m in MODEL_ORDER if m in df_all['Model'].values]).reset_index()
print(f'\n{len(df_all)} models loaded.')


  ✅  InceptionTime              Top-1=32.36%  Top-5=60.52%
  ✅  Random Forest              Top-1=20.39%  Top-5=41.75%
  ✅  Logistic Regression        Top-1=24.60%  Top-5=49.84%
  ✅  CIF                        Top-1=5.83%  Top-5=20.39%

4 models loaded.


In [6]:
# ── Table 1: APA 7 performance summary ───────────────────────────────────

def fmt(v, pct=True, dp=2):
    if v is None or (isinstance(v, float) and np.isnan(v)): return '—'
    return f'{v*100:.{dp}f}' if pct else f'{v:.{dp}f}'

cols_display = ['Model','Accuracy','Top-5 Acc','F1 Macro',
                'F1 Weighted','Precision','Recall','AUC','N Test','Model MB']
col_labels   = ['Model','Top-1\nAcc (%)','Top-5\nAcc (%)','F1 Macro\n(%)',
                'F1 Wt\n(%)','Prec\n(%)','Rec\n(%)','AUC','N Test','Size\n(MB)']

rows_fmt = []
for _, r in df_all_ordered.iterrows():
    rows_fmt.append([
        r['Model'],
        fmt(r['Accuracy']),
        fmt(r['Top-5 Acc']),
        fmt(r['F1 Macro']),
        fmt(r['F1 Weighted']),
        fmt(r['Precision']),
        fmt(r['Recall']),
        fmt(r['AUC'], pct=False),
        str(int(r['N Test'])) if r['N Test'] else '—',
        f"{r['Model MB']:.1f}" if r['Model MB'] else '—',
    ])

n_rows = len(rows_fmt)
fig_h  = 1.6 + 0.45 + n_rows * 0.38 + 0.8
fig, ax = plt.subplots(figsize=(15, fig_h))
ax.axis('off')

note_frac  = 0.8  / fig_h
title_frac = 1.6  / fig_h
table_frac = 1.0 - title_frac - note_frac
table_y0   = note_frac

tbl = ax.table(
    cellText  = rows_fmt,
    colLabels = col_labels,
    cellLoc   = 'center',
    loc       = 'center',
    bbox      = [0, table_y0, 1, table_frac]
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
for (row, col), cell in tbl.get_celld().items():
    cell.set_linewidth(0); cell.set_edgecolor('none')
    cell.set_facecolor('white')
    if row == 0: cell.set_text_props(fontweight='bold')

for y in [table_y0+table_frac,
          table_y0+table_frac*(n_rows/(n_rows+1)),
          table_y0]:
    ax.axhline(y=y, xmin=0.01, xmax=0.99, color='black', linewidth=0.8)

ax.text(0.0, 1.0-(0.12/fig_h), 'Table 1',
        transform=ax.transAxes, fontsize=10,
        fontweight='bold', ha='left', va='top')
ax.text(0.0, 1.0-(0.50/fig_h),
        'Classifier Performance on the 154-Class ASL Dataset',
        transform=ax.transAxes, fontsize=10,
        fontstyle='italic', ha='left', va='top')
ax.text(0.0, note_frac-0.02,
        'Note. All accuracy and F1 values expressed as percentages. '
        'F1 Macro = unweighted mean F1 across all 154 classes. '
        'F1 Wt = F1 weighted by class support. '
        'AUC = macro-averaged one-vs-rest ROC AUC. '
        'CIF = Canonical Interval Forest. '
        'Size = model file size on disk. Dashes indicate metric unavailable.',
        transform=ax.transAxes, fontsize=7.5, ha='left', va='top')

t1_path = os.path.join(COMPARISON_DIR, 'Table_1_154class_performance.png')
plt.savefig(t1_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved: Table_1_154class_performance.png\n')

# Also save as CSV
(
    df_all_ordered
    .drop(columns=['_y_true','_y_pred','_y_proba'], errors='ignore')
    .to_csv(os.path.join(COMPARISON_DIR,'global_comparison_154class.csv'), index=False)
)
print('Saved: global_comparison_154class.csv')


Saved: Table_1_154class_performance.png

Saved: global_comparison_154class.csv


In [7]:
# ── Figure 1: Top-1 and Top-5 Accuracy — horizontal bar ─────────────────
models_plot = [m for m in MODEL_ORDER if m in df_all['Model'].values]
top1_vals   = [df_all_ordered.set_index('Model').loc[m,'Accuracy'] for m in models_plot]
top5_vals   = [df_all_ordered.set_index('Model').loc[m,'Top-5 Acc']
               if df_all_ordered.set_index('Model').loc[m,'Top-5 Acc'] is not None
               else np.nan for m in models_plot]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
fig.subplots_adjust(wspace=0.08)

for ax_i, (vals, metric) in enumerate(zip([top1_vals, top5_vals],
                                          ['Top-1 Accuracy', 'Top-5 Accuracy'])):
    ax = axes[ax_i]
    colors = [MODEL_COLORS[m] for m in models_plot]
    bars = ax.barh(models_plot, [v if v else 0 for v in vals],
                   color=colors, edgecolor='white', linewidth=0.5,
                   alpha=0.88, height=0.55)
    ax.set_xlabel(f'{metric} (%)', fontsize=10)
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xlim(0, max(v for v in vals if v) * 1.30)
    # Labels outside bars — no overlap with data
    for bar, v in zip(bars, vals):
        if v and not np.isnan(float(v)):
            ax.text(float(v) + 0.005, bar.get_y() + bar.get_height()/2,
                    f'{float(v)*100:.1f}%',
                    va='center', ha='left', fontsize=9,
                    fontweight='bold',
                    color=MODEL_COLORS[models_plot[bars.patches.index(bar)]])

axes[0].set_title('Top-1 Accuracy\n', fontsize=11, fontweight='bold')
axes[1].set_title('Top-5 Accuracy\n', fontsize=11, fontweight='bold')
fig.suptitle('Figure 1. Classification Accuracy — 154-Class ASL\n',
             fontsize=12, fontweight='bold', y=1.02)

f1_path = os.path.join(COMPARISON_DIR, 'Figure_1_top1_top5_accuracy.png')
plt.savefig(f1_path, dpi=150, bbox_inches='tight')
plt.close()
print('Saved: Figure_1_top1_top5_accuracy.png\n')
print('Figure 1. Top-1 and Top-5 classification accuracy for each classifier '
      'on the 154-class ASL dataset. '
      'Top-5 accuracy indicates the proportion of test samples '
      'where the correct sign appeared within the five '
      'highest-ranked predictions. CIF = Canonical Interval Forest.')


Saved: Figure_1_top1_top5_accuracy.png

Figure 1. Top-1 and Top-5 classification accuracy for each classifier on the 154-class ASL dataset. Top-5 accuracy indicates the proportion of test samples where the correct sign appeared within the five highest-ranked predictions. CIF = Canonical Interval Forest.


In [8]:
# ── Figure 2: F1 Macro and F1 Weighted — grouped horizontal bar ──────────
f1mac_vals = [df_all_ordered.set_index('Model').loc[m,'F1 Macro']  for m in models_plot]
f1wt_vals  = [df_all_ordered.set_index('Model').loc[m,'F1 Weighted'] for m in models_plot]

y_pos = np.arange(len(models_plot))
h     = 0.32

fig, ax = plt.subplots(figsize=(10, 4.5))
bars_mac = ax.barh(y_pos - h/2, f1mac_vals, h,
                   color=[MODEL_COLORS[m] for m in models_plot],
                   alpha=0.90, label='F1 Macro', edgecolor='white')
bars_wt  = ax.barh(y_pos + h/2, f1wt_vals,  h,
                   color=[MODEL_COLORS[m] for m in models_plot],
                   alpha=0.55, label='F1 Weighted', edgecolor='white',
                   hatch='//')

ax.set_yticks(y_pos)
ax.set_yticklabels(models_plot, fontsize=10)
ax.set_xlabel('F1 Score (%)', fontsize=10)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
# Labels outside bars
max_val = max([v for v in f1mac_vals+f1wt_vals if v], default=0.5)
ax.set_xlim(0, max_val * 1.35)
for bar, v in zip(bars_mac, f1mac_vals):
    ax.text(float(v)+0.003, bar.get_y()+bar.get_height()/2,
            f'{float(v)*100:.1f}%', va='center', ha='left', fontsize=8)
for bar, v in zip(bars_wt, f1wt_vals):
    ax.text(float(v)+0.003, bar.get_y()+bar.get_height()/2,
            f'{float(v)*100:.1f}%', va='center', ha='left', fontsize=8)

# Legend outside data area
ax.legend(frameon=False, fontsize=9, loc='lower right')
ax.set_title('Figure 2. F1 Macro and F1 Weighted — 154-Class ASL\n',
             fontsize=11, fontweight='bold')

f2_path = os.path.join(COMPARISON_DIR, 'Figure_2_f1_macro_weighted.png')
plt.savefig(f2_path, dpi=150, bbox_inches='tight')
plt.close()
print('Saved: Figure_2_f1_macro_weighted.png\n')
print('Figure 2. Macro-averaged and weighted F1 scores for each classifier '
      'on the 154-class ASL dataset. '
      'F1 Macro weights each class equally; '
      'F1 Weighted accounts for class support. '
      'CIF = Canonical Interval Forest.')


Saved: Figure_2_f1_macro_weighted.png

Figure 2. Macro-averaged and weighted F1 scores for each classifier on the 154-class ASL dataset. F1 Macro weights each class equally; F1 Weighted accounts for class support. CIF = Canonical Interval Forest.


In [9]:
# ── Figure 3: Precision, Recall, AUC — grouped bar ──────────────────────
metrics_3   = ['Precision', 'Recall', 'AUC']
metric_lbls = ['Precision\n(Macro %)', 'Recall\n(Macro %)', 'AUC\n(Macro)']
n_metrics   = len(metrics_3)
n_models    = len(models_plot)
x3          = np.arange(n_metrics)
w3          = 0.18
offsets3    = np.linspace(-(n_models-1)/2,(n_models-1)/2,n_models)*w3

fig, ax = plt.subplots(figsize=(10, 5))
for i, model in enumerate(models_plot):
    row  = df_all_ordered.set_index('Model').loc[model]
    vals = []
    for m in metrics_3:
        v = row.get(m, None)
        vals.append(float(v) if v is not None else 0.0)

    bars = ax.bar(x3 + offsets3[i], vals, w3,
                  label=model,
                  color=MODEL_COLORS[model],
                  edgecolor='white', linewidth=0.5, alpha=0.88)
    for bar, v in zip(bars, vals):
        if v > 0:
            ax.text(bar.get_x()+bar.get_width()/2,
                    bar.get_height()+0.005,
                    f'{v:.2f}' if m=='AUC' else f'{v*100:.1f}%',
                    ha='center', va='bottom', fontsize=7,
                    color=MODEL_COLORS[model])

ax.set_xticks(x3)
ax.set_xticklabels(metric_lbls, fontsize=10)
ax.set_ylim(0, 1.15)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
# Legend placed to the right of the chart — outside data
ax.legend(frameon=False, fontsize=9,
          bbox_to_anchor=(1.01, 1), loc='upper left', borderaxespad=0)
ax.set_title('Figure 3. Precision, Recall, and AUC — 154-Class ASL\n',
             fontsize=11, fontweight='bold')

f3_path = os.path.join(COMPARISON_DIR, 'Figure_3_precision_recall_auc.png')
plt.savefig(f3_path, dpi=150, bbox_inches='tight')
plt.close()
print('Saved: Figure_3_precision_recall_auc.png\n')
print('Figure 3. Macro-averaged precision, recall, and AUC for each '
      'classifier on the 154-class ASL dataset. '
      'AUC = macro one-vs-rest ROC AUC. '
      'CIF = Canonical Interval Forest.')


Saved: Figure_3_precision_recall_auc.png

Figure 3. Macro-averaged precision, recall, and AUC for each classifier on the 154-class ASL dataset. AUC = macro one-vs-rest ROC AUC. CIF = Canonical Interval Forest.


In [10]:
print('=' * 65)
print('  OUTPUT FILE MANIFEST')
print('=' * 65)
print(f'  Directory: {COMPARISON_DIR}\n')
all_outputs = sorted(os.listdir(COMPARISON_DIR))
tables  = [f for f in all_outputs if f.startswith('Table')]
figures = [f for f in all_outputs if f.startswith('Figure')]
csvs    = [f for f in all_outputs if f.endswith('.csv')]

print('  Tables (APA 7):')
for f in tables:
    kb = os.path.getsize(os.path.join(COMPARISON_DIR,f))//1024
    print(f'    {f}  ({kb} KB)')
print()
print('  Figures:')
for f in figures:
    kb = os.path.getsize(os.path.join(COMPARISON_DIR,f))//1024
    print(f'    {f}  ({kb} KB)')
print()
print('  Data files:')
for f in csvs:
    kb = os.path.getsize(os.path.join(COMPARISON_DIR,f))//1024
    print(f'    {f}  ({kb} KB)')
print()
print('Global comparison complete.')


  OUTPUT FILE MANIFEST
  Directory: /content/drive/MyDrive/Consultant/Colab_Notebooks/Obrown_Dissertation_NU_25/OBrown_DIS9300_v2/results/end_point/global_comparison_v2

  Tables (APA 7):
    Table_1_154class_performance.png  (87 KB)

  Figures:
    Figure_1_top1_top5_accuracy.png  (60 KB)
    Figure_2_f1_macro_weighted.png  (53 KB)
    Figure_3_precision_recall_auc.png  (51 KB)

  Data files:
    global_comparison_154class.csv  (0 KB)

Global comparison complete.
